# Lesson 16: Modern upgrades

**Why:** our GPT has the right **architecture**. Real GPTs also use a handful of **engineering tricks** that make training faster, steadier and a bit better. None of them change the core ideas; they're refinements. We add six, all switched on together with `--modern`:

| Upgrade | What it does | Plain-English version |
|---|---|---|
| **Fast attention** | all heads at once, using PyTorch's built-in attention | the same math as Lessons 8–10, without the slow per-head loop |
| **Weight tying** | output layer shares the token table | "reading" a word and "writing" it use the same knowledge |
| **GPT-2 initialization** | small, carefully sized starting values | start calm, so stacking blocks doesn't blow numbers up |
| **Warm-up + cosine learning rate** | small steps at first, big in the middle, small at the end | ease in, cruise, then slow down to settle precisely |
| **Weight decay** | gently pulls big weights toward 0 | discourages memorizing (like dropout, a different way) |
| **Gradient clipping** | caps very large updates | one weird batch can't knock the model off course |

The model and training code are in [`model.py`](../src/tinygpt/model.py) and [`train.py`](../src/tinygpt/train.py). The upgraded model trained in the background with **the same tokenizer, size and 3,000 steps** as Lesson 15, so the comparison is fair.

## 1. The learning-rate schedule

Lesson 2 showed that the step size makes or breaks training. Instead of one fixed rate, we **warm up** over the first 100 steps (fresh, random models make wild early updates), then **glide down a cosine curve** so the last steps are small, careful refinements.

In [ ]:
import math
import time
from types import SimpleNamespace
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from tinygpt.model import GPT, GPTConfig
from tinygpt.train import CHECKPOINT_DIR, get_lr, load_model

plan = SimpleNamespace(modern=True, lr=1e-3, min_lr=1e-4, warmup_steps=100, max_steps=3000)
steps = range(3000)
plt.figure(figsize=(8, 3))
plt.plot(steps, [1e-3] * 3000, "--", label="Lessons 12 & 15: fixed")
plt.plot(steps, [get_lr(s, plan) for s in steps], label="Lesson 16: warm-up + cosine")
plt.xlabel("step"); plt.ylabel("learning rate"); plt.legend(); plt.title("Step size over training"); plt.show()

## 2. Fast attention: same answers, less waiting

In Lessons 8–10 each head was its own little module, run one after the other in a Python loop. The fast version makes **one** big `Linear` for all queries, keys and values, turns the heads into an extra **batch dimension**, and lets PyTorch's optimized `F.scaled_dot_product_attention` do the q·k, ÷√hs, mask, softmax and ·v in a single call:

```
x (B, T, C) ─► one Linear ─► q, k, v (B, T, C) each ─► split into heads ─► (B, nh, T, hs)
            ─► scaled_dot_product_attention(causal) ─► (B, nh, T, hs) ─► join heads ─► (B, T, C)
```

The test suite now proves the two give **identical results** with the same weights (`test_fast_attention_matches_hand_built_attention`).

**Predict:** at our model size, how much faster will a training step be: about **1.1×**, **1.5×** or **5×**?

In [ ]:
def ms_per_step(config, steps=8):
    torch.manual_seed(0)
    model = GPT(config)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    x = torch.randint(config.vocab_size, (32, config.block_size))
    for i in range(steps + 2):
        if i == 2:
            start = time.perf_counter()
        _, loss = model(x, x)
        opt.zero_grad(); loss.backward(); opt.step()
    return (time.perf_counter() - start) / steps * 1000

base = GPTConfig(vocab_size=512, block_size=128, n_embd=128, n_head=4, n_layer=4, dropout=0.1)
slow = ms_per_step(base)
fast = ms_per_step(GPTConfig(**{**base.__dict__, "fast_attention": True}))
print(f"one head at a time: {slow:.0f} ms per step")
print(f"all heads at once:  {fast:.0f} ms per step   ({slow / fast:.1f}x faster)")

## 3. Weight tying and GPT-2 initialization

**Weight tying:** the token table (V × C) turns a token into numbers; the output layer (C × V) turns numbers back into token scores. They're two views of the same knowledge ("what is this token like?"), so modern GPTs **share one table** for both.

**GPT-2 initialization:** every weight starts small (spread 0.02), and the layers that **add onto the residual path** start even smaller (÷ √(2 × layers)), so 4 or 40 stacked blocks all start calm. The starting loss should land right at the "knows nothing" value, ln(512) = 6.24.

In [ ]:
modern = GPTConfig(**{**base.__dict__, "fast_attention": True, "tie_weights": True, "gpt2_init": True})
x = torch.randint(512, (32, 128))
y = torch.randint(512, (32, 128))   # random answers, like the very first training step
for name, config in [("Lesson 15 style", base), ("modern", modern)]:
    torch.manual_seed(0)
    model = GPT(config).eval()
    with torch.no_grad():
        loss = model(x, y)[1].item()
    print(f"{name:16} {model.num_params():,} parameters   starting loss {loss:.3f}   (knows nothing: {math.log(512):.3f})")

## 4. Inside the upgraded training run

The training script now records two extra things at every check: the **learning rate** in use, and the average **gradient size** (how big a nudge the model wanted, before clipping caps it at 1.0).

In [ ]:
runs = {}
for name, file in [("Lesson 15 BPE", "tesla_bpe_best.pt"), ("Lesson 16 modern", "tesla_modern_best.pt")]:
    model, tok, ckpt = load_model(CHECKPOINT_DIR / file)
    runs[name] = {"model": model, "tok": tok, "ckpt": ckpt}

h = runs["Lesson 16 modern"]["ckpt"]["history"]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.5))
a1.plot([r["step"] for r in h], [r["lr"] for r in h], "o-"); a1.set_title("learning rate used"); a1.set_xlabel("step")
norms = [(r["step"], r["grad_norm"]) for r in h if r["grad_norm"] is not None]
a2.plot(*zip(*norms), "o-", color="tab:orange"); a2.axhline(1.0, ls="--", color="gray", label="clipping limit")
a2.set_title("average gradient size"); a2.set_xlabel("step"); a2.legend()
plt.show()

## 5. Did it help? The fair comparison

Bits per character on the same held-out text (Lesson 15's method), the learning curves against training minutes, and both models' writing.

In [ ]:
from tinygpt.data import get_batch, load_text

text = load_text()
val_text = text[int(0.9 * len(text)):]

@torch.no_grad()
def bits_per_char(model, tok, text, batches=100):
    ids = torch.tensor(tok.encode(text))
    chars_per_token = len(text) / len(ids)
    torch.manual_seed(0)
    loss = sum(model(*get_batch(ids, 32, model.config.block_size))[1].item() for _ in range(batches)) / batches
    return loss / chars_per_token / math.log(2), chars_per_token

plt.figure(figsize=(9, 4))
for name, r in runs.items():
    r["bpc"], cpt = bits_per_char(r["model"], r["tok"], val_text)
    hist = r["ckpt"]["history"]
    plt.plot([x["minutes"] for x in hist], [x["val"] / cpt / math.log(2) for x in hist], "o-", label=name)
    print(f"{name:17} {r['bpc']:.3f} bits per character   ({r['ckpt']['minutes']:.0f} min of training, "
          f"final train/val gap {hist[-1]['val'] - hist[-1]['train']:.2f})")
plt.xlabel("minutes of training"); plt.ylabel("validation bits per character"); plt.ylim(1.4, 3.0)
plt.legend(); plt.title("Same model size, same steps, with and without the upgrades"); plt.show()

In [ ]:
prompt = "The alternating current "
for name, r in runs.items():
    torch.manual_seed(42)
    out = r["model"].generate(torch.tensor([r["tok"].encode(prompt)]), 250, temperature=0.8)
    print(f"===== {name} =====")
    print(r["tok"].decode(out[0].tolist()))
    print()

## 6. What newer LLMs changed (for reference)

Our GPT follows GPT-2 (2019). Models like Llama, Mistral and Claude-class systems keep the same skeleton (embeddings → stacked blocks of attention + feed-forward with residuals → output), and swap a few parts:

| Our GPT-2-style part | Common modern replacement | Why |
|---|---|---|
| LayerNorm | **RMSNorm** | simpler (no centring), slightly faster, works as well |
| learned position embeddings | **RoPE** (rotary positions) | encodes *relative* distance inside attention; stretches to longer texts |
| ReLU feed-forward | **SwiGLU** | a "gated" feed-forward that learns a bit better per parameter |
| every head has its own k, v | **grouped-query attention** | heads share keys/values, so writing long answers needs much less memory |
| 128 tokens of context | 100,000+ tokens | more efficient attention, huge GPUs |
| 1 MB of text, 40 min on a laptop | trillions of tokens, months on thousands of GPUs | scale is most of the difference |

After that comes a different stage entirely: **fine-tuning** on conversations and **learning from human feedback**, which turns a text-continuer like ours into an assistant that follows instructions.

## 7. Final check

In [ ]:
assert fast < slow, "fast attention should be faster"
assert all(0.5 < r["bpc"] < 3 for r in runs.values())
print("Lesson 16 complete.")

## Wrap-up question

Which of the six upgrades do you think would matter **most** if we made the model 100× bigger and trained it for days? Why?